# 🛡️ ElectroMA WhatsApp AI Agent (RELIA) — Production & Hardening Guide

Comprehensive audit, architecture recommendations, and step-by-step code snippets to make your WhatsApp AI Agent bulletproof in production.

---

## 1. Webhook Architecture (FastAPI BackgroundTasks)

### Why it matters:
Meta's WhatsApp Cloud API requires your `/webhook` endpoint to return HTTP `200 OK` within **5 seconds**.
If processing takes longer, Meta will retry sending duplicate messages.

### Solution Code:

In [ ]:
# main.py snippet
from fastapi import FastAPI, Request, BackgroundTasks
from langchain_core.messages import HumanMessage

app = FastAPI()

def process_and_reply(phone: str, text: str):
    try:
        result = agent.invoke(
            {"messages": [HumanMessage(content=f"[Customer WhatsApp: {phone}]\n{text}")]},
            config={"configurable": {"thread_id": phone}}
        )
        reply = result["messages"][-1].content
        _send(phone, reply)
    except Exception as e:
        logger.error(f"Background task failed for {phone}: {e}", exc_info=True)
        _send(phone, "Désolé, problème technique. Réessayez plus tard.")

@app.post("/webhook")
async def receive(request: Request, background_tasks: BackgroundTasks):
    data = await request.json()
    msg_id = _get_msg_id(data)
    if msg_id and msg_id in _processed:
        return {"status": "duplicate"}
    if msg_id:
        _processed.add(msg_id)

    try:
        entry = data["entry"][0]["changes"][0]["value"]
        if "messages" not in entry:
            return {"status": "ignored"}
        msg = entry["messages"][0]
        phone = msg["from"]
        
        if msg.get("type") == "text":
            text = msg["text"]["body"]
            # Return 200 OK immediately and process in background
            background_tasks.add_task(process_and_reply, phone, text)
            return {"status": "queued"}
    except Exception:
        pass
    return {"status": "ok"}

## 2. Database & Concurrency (SQLite WAL Mode)

Enable Write-Ahead Logging (WAL) and set a 5-second busy timeout to prevent `database is locked` errors during concurrent customer messages.

In [ ]:
# agent.py snippet
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("memory.db", check_same_thread=False)
conn.execute("PRAGMA journal_mode=WAL;")
conn.execute("PRAGMA busy_timeout=5000;")
conn.commit()

memory = SqliteSaver(conn)

## 3. WhatsApp Morocco Edge Cases

1. **Audio Voice Notes**: Auto-reply requesting text or transcribe with Whisper / Gemini API.
2. **Location Pins**: Parse `msg['location']` coordinates into a Google Maps URL for delivery.
3. **Currency & Price Synonyms**: Prompt mapped `dh`, `DH`, `dhs`, `dirham`, `درهم`, and bare numbers to `MAD`.

## 4. OpenRouter Model Naming

Always use the provider prefix on OpenRouter:
- `google/gemini-2.0-flash-001`
- `google/gemini-2.5-flash-lite`
- `openai/gpt-4o-mini`